Setup Clearml

In [1]:
pip install clearml

Note: you may need to restart the kernel to use updated packages.


In [35]:
%env CLEARML_WEB_HOST=https://app.5ccsagap.er.kcl.ac.uk/
%env CLEARML_API_HOST=https://api.5ccsagap.er.kcl.ac.uk
%env CLEARML_FILES_HOST=https://files.5ccsagap.er.kcl.ac.uk
# GAP_group3_embedding
%env CLEARML_API_ACCESS_KEY=F3JFSR84AVXM6OTRRRUBQ5QPVHMTWV
%env CLEARML_API_SECRET_KEY=tda_99OfnqWYSguFYS5wH6FtKuVBoryYeREQVh2MPdcXct0hGiNEnfxeIOBpXPH1fcE

env: CLEARML_WEB_HOST=https://app.5ccsagap.er.kcl.ac.uk/
env: CLEARML_API_HOST=https://api.5ccsagap.er.kcl.ac.uk
env: CLEARML_FILES_HOST=https://files.5ccsagap.er.kcl.ac.uk
env: CLEARML_API_ACCESS_KEY=F3JFSR84AVXM6OTRRRUBQ5QPVHMTWV
env: CLEARML_API_SECRET_KEY=tda_99OfnqWYSguFYS5wH6FtKuVBoryYeREQVh2MPdcXct0hGiNEnfxeIOBpXPH1fcE


Load Embedding Model

In [43]:
from openai import OpenAI
from dotenv import load_dotenv
import os
 
load_dotenv()
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def openai_small(text):
    response = client.embeddings.create(
        input=[text],
        model="text-embedding-3-small"
    )
    return response.data[0].embedding  

def openai_large(text):
    response = client.embeddings.create(
        input=[text],
        model="text-embedding-3-large"
    )
    return response.data[0].embedding   


In [ ]:
pip install -U sentence-transformers

In [45]:
from sentence_transformers import SentenceTransformer
 
hf_mini_model = SentenceTransformer("all-MiniLM-L6-v2")   
hf_mpnet_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def hf_mini(text):
    return hf_mini_model.encode(text).tolist()

def hf_mpnet(text):
    return hf_mpnet_model.encode(text).tolist()

Define cosine similarity

In [23]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(vec1, vec2):
    vec1 = np.array(vec1).reshape(1, -1)
    vec2 = np.array(vec2).reshape(1, -1)
    return cosine_similarity(vec1, vec2)[0][0]

install MTEB

In [ ]:
pip install mteb

In [3]:
from mteb import MTEB

Semantic similarity

In [ ]:
from datasets import load_dataset
 
dataset = load_dataset("stsb_multi_mt", name="en", split="test")
 
print(dataset[0])


Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/1500 [00:00<?, ? examples/s]

{'sentence1': 'A girl is styling her hair.', 'sentence2': 'A girl is brushing her hair.', 'similarity_score': 2.5}


In [ ]:
import random
# randomly sample 50 from the dataset
random.seed(42)   
small_benchmark = dataset.shuffle(seed=42).select(range(50))


In [46]:
def evaluate_on_mteb(dataset, embed_function):
    similarities = []
    human_scores = []

    for example in dataset:
        sent1, sent2, score = example["sentence1"], example["sentence2"], example["similarity_score"]
        
        emb1 = embed_function(sent1)
        emb2 = embed_function(sent2)

        model_similarity = cosine_sim(emb1, emb2)

        similarities.append(model_similarity)
        human_scores.append(score / 5.0)   

    from scipy.stats import pearsonr
    correlation, _ = pearsonr(similarities, human_scores)
    
    return correlation
 
hf_mini_correlation = evaluate_on_mteb(small_benchmark, hf_mini)
hf_mpnet_correlation = evaluate_on_mteb(small_benchmark, hf_mpnet) 
openai_small_correlation = evaluate_on_mteb(small_benchmark, openai_small)
openai_large_correlation = evaluate_on_mteb(small_benchmark, openai_large)

print(f"OpenAI Small Model Pearson Correlation: {openai_small_correlation:.4f}")
print(f"OpenAI Large Model Pearson Correlation: {openai_large_correlation:.4f}")
print(f"hf_mini_correlation Pearson Correlation: {hf_mini_correlation:.4f}")
print(f"hf_mpnet_correlation Pearson Correlation: {hf_mpnet_correlation:.4f}")

OpenAI Small Model Pearson Correlation: 0.8672
OpenAI Large Model Pearson Correlation: 0.8668
hf_mini_correlation Pearson Correlation: 0.7993
hf_mpnet_correlation Pearson Correlation: 0.8423


In [47]:
from clearml import Task 
task = Task.init(project_name="MTEB Benchmark", task_name="Evaluate Embedding Models Semantic Similarity")
 
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "OpenAI Small Model", openai_small_correlation, iteration=1)
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "OpenAI Large Model", openai_large_correlation, iteration=1)
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "HF mini Model", hf_mini_correlation, iteration=1) 
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "HF mpnet Model", hf_mpnet_correlation, iteration=1)

task.close()


ClearML Task: created new task id=2500cf2e75ae495c8f719f6e899a953e
ClearML results page: https://app.5ccsagap.er.kcl.ac.uk/projects/0aa82f037e9742359c811b087eff9e2f/experiments/2500cf2e75ae495c8f719f6e899a953e/output/log
